In [ ]:
import pandas as pd
import sqlite3

df_clean = pd.read_csv('../data/processed/telco_churn_clean.csv')
conn = sqlite3.connect('../data/processed/telco_churn.db')
df_clean.to_sql('customers', conn, if_exists='replace', index=False)

#¿Cuál es la tasa de clientes perdidos?
query = """
SELECT
    COUNT(*) AS total_clientes,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS clientes_perdidos,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS tasa_churn_porcentaje
FROM customers;
"""
pd.read_sql_query(query, conn)

,total_clientes,clientes_perdidos,tasa_churn_porcentaje
0,7043,1869,26.54


In [2]:
#¿Cómo afecta el tipo de contrato (mes a mes vs. anual) a la fuga de clientes?
q_contrato = """
SELECT
    Contract AS tipo_contrato,
    COUNT(*) AS total_clientes,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS bajas,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS tasa_churn_pct
FROM customers
GROUP BY Contract
ORDER BY tasa_churn_pct DESC;
"""

#¿Qué sucede en los primeros meses?
q_antiguedad = """
SELECT
    TenureGroup AS antiguedad,
    COUNT(*) AS total_clientes,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS bajas,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS tasa_churn_pct
FROM customers
GROUP BY TenureGroup
ORDER BY tasa_churn_pct DESC;
"""

#¿Qué servicio tiene más bajas?
q_internet = """
SELECT
    InternetService AS servicio_internet,
    COUNT(*) AS total_clientes,
    ROUND(AVG(MonthlyCharges), 2) AS factura_mensual_promedio,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS tasa_churn_pct
FROM customers
GROUP BY InternetService
ORDER BY tasa_churn_pct DESC;
"""

#¿El soporte técnico reduce las bajas en clientes con fibra óptica?
q_soporte = """
SELECT
    TechSupport AS tiene_soporte_tecnico,
    COUNT(*) AS total_clientes,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS bajas,
    ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS tasa_churn_pct
FROM customers
GROUP BY TechSupport
ORDER BY tasa_churn_pct DESC;
"""

print("--- 1. Por contrato ---")
display(pd.read_sql_query(q_contrato, conn))

print("--- 2. Por antigüedad ---")
display(pd.read_sql_query(q_antiguedad, conn))

print("--- 3. Por servicio ---")
display(pd.read_sql_query(q_internet, conn))

print("--- 4. Por soporte ---")
display(pd.read_sql_query(q_soporte, conn))

--- 1. Por contrato ---


,tipo_contrato,total_clientes,bajas,tasa_churn_pct
0,Month-to-month,3875,1655,42.71
1,One year,1473,166,11.27
2,Two year,1695,48,2.83


--- 2. Por antigüedad ---


,antiguedad,total_clientes,bajas,tasa_churn_pct
0,0-6 meses,1481,784,52.94
1,6-12 meses,705,253,35.89
2,1-2 años,1024,294,28.71
3,+2 años,3833,538,14.04


--- 3. Por servicio ---


,servicio_internet,total_clientes,factura_mensual_promedio,tasa_churn_pct
0,Fiber optic,3096,91.50,41.89
1,DSL,2421,58.10,18.96
2,No,1526,21.08,7.40


--- 4. Por soporte ---


,tiene_soporte_tecnico,total_clientes,bajas,tasa_churn_pct
0,No,3473,1446,41.64
1,Yes,2044,310,15.17
2,No internet service,1526,113,7.40
